# Normalization and Design

## Quick notes
### What this covers
How to decide which tables and columns you need: normal forms (1NF, 2NF, 3NF, BCNF), the problems bad design causes (anomalies), keys, relationships, and practical column-type choices (money, dates, enums, JSON). Ends with a worked schema design.
### Key points
- Normalization = store each fact in exactly ONE place.
- 1NF: one value per cell, no repeating columns. 2NF: every non-key column depends on the whole key. 3NF: non-key columns depend on the key, not on other non-key columns. BCNF: every (non-trivial) determinant is a key.
- Bad design causes update, insert and delete anomalies.
- Relationships: one-to-many = foreign key on the "many" side. Many-to-many = a bridge (junction) table.
- Store money as integer cents, dates as ISO-8601 text (`YYYY-MM-DD`), small fixed lists as a lookup table or a CHECK.
- Denormalize only on purpose, for measured read speed, and keep the source of truth clear.
### Gotchas (the part worth revising twice)
- "Normalize until it hurts, denormalize until it works" is a joke, but a real trade-off: more joins versus safer updates.
- Comma-separated lists in one column (`'red,green'`) break 1NF and make search and joins painful.
- Natural keys (email, phone) change. Surrogate keys (`id`) are stable. Still add a UNIQUE on the natural key.
- `REAL` for money gives rounding errors (0.1 + 0.2 is not 0.3). Use integer cents.
- SQLite does not enforce column types strictly unless the table is `STRICT`; add CHECKs for the rules you care about.
### When it matters
Before you write the first CREATE TABLE. Fixing a schema after data exists is far more expensive than getting it roughly right at the start.
### One-line recap example
```sql
-- Do not repeat the customer on every order line: store it once and point to it.
orders(id, customer_id REFERENCES customers(id))
```

## 01. Why design matters: the three anomalies
- Imagine ONE big table with customer, product and order facts mixed together.
- **Update anomaly**: one fact is stored in many rows. Change it in some rows and miss others: the data now disagrees with itself.
- **Insert anomaly**: you cannot record one fact without another (a new product cannot be saved until someone orders it).
- **Delete anomaly**: deleting one fact destroys another (deleting the only order for a product also deletes the product).
- Normalization is the cure. The next cells show each anomaly with real rows.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE orders_wide (
    order_id       INTEGER NOT NULL,
    product_sku    TEXT    NOT NULL,
    customer_name  TEXT    NOT NULL,
    customer_email TEXT    NOT NULL,
    product_name   TEXT    NOT NULL,
    unit_price     INTEGER NOT NULL,   -- cents
    quantity       INTEGER NOT NULL,
    PRIMARY KEY (order_id, product_sku)
);
INSERT INTO orders_wide VALUES
    (1, 'SKU-1', 'Alice', 'alice@example.com', 'Widget',  999, 2),
    (1, 'SKU-2', 'Alice', 'alice@example.com', 'Gadget', 1999, 1),
    (2, 'SKU-1', 'Bob',   'bob@example.com',   'Widget',  999, 5),
    (3, 'SKU-2', 'Alice', 'alice@example.com', 'Gadget', 1999, 3);
SELECT * FROM orders_wide ORDER BY order_id, product_sku;

order_id | product_sku | customer_name | customer_email    | product_name | unit_price | quantity
---------+-------------+---------------+-------------------+--------------+------------+---------
1        | SKU-1       | Alice         | alice@example.com | Widget       | 999        | 2       
1        | SKU-2       | Alice         | alice@example.com | Gadget       | 1999       | 1       
2        | SKU-1       | Bob           | bob@example.com   | Widget       | 999        | 5       
3        | SKU-2       | Alice         | alice@example.com | Gadget       | 1999       | 3       
(4 rows)



## 02. Update anomaly, shown
- Alice changes her email. A careless `UPDATE` with the wrong filter fixes only one order.
- Now the same customer has two different emails. Which one is true? The database cannot tell you.

In [3]:
%%sql
UPDATE orders_wide SET customer_email = 'alice@new.example' WHERE order_id = 1 AND product_sku = 'SKU-1';   -- only one row fixed
SELECT DISTINCT customer_name, customer_email FROM orders_wide WHERE customer_name = 'Alice' ORDER BY customer_email;

customer_name | customer_email   
--------------+------------------
Alice         | alice@example.com
Alice         | alice@new.example
(2 rows)



## 03. Insert and delete anomalies, shown
- Insert: we want to add a new product 'Gizmo' to the catalogue. There is no order yet, so there is no row to hold it. We would have to invent a fake order or leave NOT NULL columns empty.
- Delete: Bob cancels order 2. That was the only row for his email, so we lose all knowledge of Bob. Deleting order lines can also delete the only copy of a product's details.

In [4]:
%%sql expect_error
-- Insert anomaly: cannot store a product alone (order and customer columns are required)
INSERT INTO orders_wide (product_sku, product_name, unit_price) VALUES ('SKU-3', 'Gizmo', 2999);

ERROR: NOT NULL constraint failed: orders_wide.order_id



In [5]:
%%sql
-- Delete anomaly: removing Bob's only order also removes every trace of Bob
DELETE FROM orders_wide WHERE order_id = 2;
SELECT count(*) AS rows_about_bob FROM orders_wide WHERE customer_name = 'Bob';

rows_about_bob
--------------
0             
(1 row)



## 04. Functional dependencies in plain words
- "A determines B" (written A -> B) means: if you know A, there is exactly one possible B.
- `customer_id -> customer_email`: one customer id, one email.
- `sku -> product_name, unit_price`: one SKU, one name and price.
- `(order_id, sku) -> quantity`: you need BOTH to know the quantity.
- The key of a table is the smallest set that determines every other column. Normal forms are rules about which dependencies are allowed.
- Trick to test a guess: look for two rows with the same A but different B. If the data allows it, A does not determine B.

In [6]:
%%sql
-- Test "does customer_name determine customer_email?" in the data: any name with more than one email?
SELECT customer_name, count(DISTINCT customer_email) AS emails
FROM orders_wide GROUP BY customer_name HAVING emails > 1 ORDER BY customer_name;

customer_name | emails
--------------+-------
Alice         | 2     
(1 row)



## 05. 1NF: first normal form
- Every cell holds ONE atomic value. No lists in a cell. No repeating columns (`phone1`, `phone2`, `phone3`).
- Every row can be identified by a primary key.
- Bad: `skills = 'sql,python,excel'`. You cannot search, count, join, or constrain a list inside a string.
- Good: a separate row per value (a child table).

In [7]:
%%sql
-- Bad: a list hiding in one cell
CREATE TABLE emp_bad (id INTEGER PRIMARY KEY, name TEXT, skills TEXT);
INSERT INTO emp_bad VALUES (1, 'Asha', 'sql,python'), (2, 'Ben', 'excel,sql');
-- "who knows sql?" needs string tricks, and 'sqlite' would match LIKE '%sql%' by mistake
SELECT name FROM emp_bad WHERE skills LIKE '%sql%' ORDER BY name;

name
----
Asha
Ben 
(2 rows)



In [8]:
%%sql
-- Good: one row per skill
CREATE TABLE emp (id INTEGER PRIMARY KEY, name TEXT NOT NULL);
CREATE TABLE emp_skill (emp_id INTEGER NOT NULL REFERENCES emp(id), skill TEXT NOT NULL, PRIMARY KEY (emp_id, skill));
INSERT INTO emp VALUES (1, 'Asha'), (2, 'Ben');
INSERT INTO emp_skill VALUES (1, 'sql'), (1, 'python'), (2, 'excel'), (2, 'sql');
SELECT e.name FROM emp e JOIN emp_skill s ON s.emp_id = e.id WHERE s.skill = 'sql' ORDER BY e.name;
SELECT skill, count(*) AS people FROM emp_skill GROUP BY skill ORDER BY skill;

name
----
Asha
Ben 
(2 rows)

skill  | people
-------+-------
excel  | 1     
python | 1     
sql    | 2     
(3 rows)



## 06. 2NF: second normal form
- Must be in 1NF. Applies when the key has more than one column.
- Every non-key column must depend on the WHOLE key, not on just part of it.
- In `orders_wide` the key is `(order_id, product_sku)`. But `customer_name` and `customer_email` depend only on `order_id`, and `unit_price` only on `product_sku`. Those are partial dependencies: 2NF violations.
- Fix: move each fact to the table whose key it depends on completely.

## 07. 3NF: third normal form
- Must be in 2NF. No TRANSITIVE dependencies: a non-key column must depend on the key, not on another non-key column.
- Bad: `employee(id, dept_id, dept_name)`. `id -> dept_id -> dept_name`. Rename the department and you must touch every employee row.
- Good: `department(id, name)` and `employee(id, dept_id)`.
- Memory line: "every non-key column depends on the key, the whole key, and nothing but the key."

In [9]:
%%sql
CREATE TABLE employee_bad (id INTEGER PRIMARY KEY, name TEXT, dept_id INTEGER, dept_name TEXT);
INSERT INTO employee_bad VALUES (1, 'Asha', 10, 'Data'), (2, 'Ben', 10, 'Data'), (3, 'Chitra', 20, 'Design');
-- rename the Data department: miss one row and you have two names for department 10
UPDATE employee_bad SET dept_name = 'Analytics' WHERE id = 1;
SELECT dept_id, dept_name FROM employee_bad ORDER BY id;

dept_id | dept_name
--------+----------
10      | Analytics
10      | Data     
20      | Design   
(3 rows)



In [10]:
%%sql
CREATE TABLE department (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE);
CREATE TABLE employee (id INTEGER PRIMARY KEY, name TEXT NOT NULL, dept_id INTEGER NOT NULL REFERENCES department(id));
INSERT INTO department VALUES (10, 'Data'), (20, 'Design');
INSERT INTO employee VALUES (1, 'Asha', 10), (2, 'Ben', 10), (3, 'Chitra', 20);
UPDATE department SET name = 'Analytics' WHERE id = 10;    -- one row, always consistent
SELECT e.name, d.name AS dept FROM employee e JOIN department d ON d.id = e.dept_id ORDER BY e.id;

name   | dept     
-------+----------
Asha   | Analytics
Ben    | Analytics
Chitra | Design   
(3 rows)



## 08. BCNF: Boyce-Codd normal form
- A stricter 3NF: for every non-trivial dependency A -> B, A must be a key (a "determinant" must be a key).
- 3NF still allows a gap when a non-key column determines PART of a key.
- Example: students pick a course and are taught by a teacher. Rule: each teacher teaches only ONE course. Key is `(student, course)`, but `teacher -> course`, and teacher is not a key. 3NF accepts it, BCNF does not.
- Symptom: the same teacher-course pair is repeated for every student, and you can change one copy and not the others.
- Fix: split into `teacher_course(teacher PK, course)` and `student_teacher(student, teacher)`.
- Trade-off: BCNF decomposition can lose the ability to enforce some multi-column rules with a simple constraint. In practice 3NF is the usual target; use BCNF when you see this pattern.

In [11]:
%%sql
CREATE TABLE enrol_bad (
    student TEXT NOT NULL, course TEXT NOT NULL, teacher TEXT NOT NULL,
    PRIMARY KEY (student, course)
);
INSERT INTO enrol_bad VALUES ('Asha', 'Maths', 'Mr Rao'), ('Ben', 'Maths', 'Mr Rao'), ('Asha', 'Art', 'Ms Lee');
-- teacher -> course holds, but nothing stops this contradiction (Mr Rao teaching Art too):
INSERT INTO enrol_bad VALUES ('Chitra', 'Art', 'Mr Rao');
SELECT teacher, count(DISTINCT course) AS courses FROM enrol_bad GROUP BY teacher ORDER BY teacher;

teacher | courses
--------+--------
Mr Rao  | 2      
Ms Lee  | 1      
(2 rows)



In [12]:
%%sql
-- BCNF version: the rule "a teacher has one course" is now enforced by the primary key
CREATE TABLE teacher_course (teacher TEXT PRIMARY KEY, course TEXT NOT NULL);
CREATE TABLE student_teacher (
    student TEXT NOT NULL,
    teacher TEXT NOT NULL REFERENCES teacher_course(teacher),
    PRIMARY KEY (student, teacher)
);
INSERT INTO teacher_course VALUES ('Mr Rao', 'Maths'), ('Ms Lee', 'Art');
INSERT INTO student_teacher VALUES ('Asha', 'Mr Rao'), ('Ben', 'Mr Rao'), ('Asha', 'Ms Lee');
SELECT st.student, tc.course, st.teacher
FROM student_teacher st JOIN teacher_course tc ON tc.teacher = st.teacher ORDER BY st.student, tc.course;

student | course | teacher
--------+--------+--------
Asha    | Art    | Ms Lee 
Asha    | Maths  | Mr Rao 
Ben     | Maths  | Mr Rao 
(3 rows)



In [13]:
%%sql expect_error
-- The contradiction is now impossible to store
INSERT INTO teacher_course VALUES ('Mr Rao', 'Art');

ERROR: UNIQUE constraint failed: teacher_course.teacher



## 09. Before and after: normalizing the order table
- Split `orders_wide` into four tables, each fact stored once: customers, products, orders, order_items.
- `unit_price` moves to `products` (current price), and we also keep the price paid on the line (`price_at_sale`), because history must not change when prices change. This is a deliberate, correct "copy" of a fact: the price at that moment is a different fact from today's price.
- See 02-Querying/01-Joins (join basics) for rebuilding the wide view.

In [14]:
%%sql
CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT NOT NULL, email TEXT NOT NULL UNIQUE);
CREATE TABLE products  (sku TEXT PRIMARY KEY, name TEXT NOT NULL, price INTEGER NOT NULL CHECK (price >= 0));
CREATE TABLE orders    (id INTEGER PRIMARY KEY, customer_id INTEGER NOT NULL REFERENCES customers(id), placed_on TEXT NOT NULL);
CREATE TABLE order_items (
    order_id      INTEGER NOT NULL REFERENCES orders(id),
    sku           TEXT    NOT NULL REFERENCES products(sku),
    quantity      INTEGER NOT NULL CHECK (quantity > 0),
    price_at_sale INTEGER NOT NULL,
    PRIMARY KEY (order_id, sku)
);
INSERT INTO customers VALUES (1, 'Alice', 'alice@example.com'), (2, 'Bob', 'bob@example.com');
INSERT INTO products VALUES ('SKU-1', 'Widget', 999), ('SKU-2', 'Gadget', 1999), ('SKU-3', 'Gizmo', 2999);   -- Gizmo needs no order: no insert anomaly
INSERT INTO orders VALUES (1, 1, '2025-01-05'), (2, 2, '2025-01-09'), (3, 1, '2025-02-01');
INSERT INTO order_items VALUES (1, 'SKU-1', 2, 999), (1, 'SKU-2', 1, 1999), (2, 'SKU-1', 5, 999), (3, 'SKU-2', 3, 1999);
SELECT c.name, o.id AS order_id, p.name AS product, oi.quantity, oi.price_at_sale
FROM order_items oi
JOIN orders o ON o.id = oi.order_id
JOIN customers c ON c.id = o.customer_id
JOIN products p ON p.sku = oi.sku
ORDER BY o.id, p.sku;

name  | order_id | product | quantity | price_at_sale
------+----------+---------+----------+--------------
Alice | 1        | Widget  | 2        | 999          
Alice | 1        | Gadget  | 1        | 1999         
Bob   | 2        | Widget  | 5        | 999          
Alice | 3        | Gadget  | 3        | 1999         
(4 rows)



In [15]:
%%sql
-- The anomalies are gone
UPDATE customers SET email = 'alice@new.example' WHERE id = 1;      -- one row, consistent everywhere
SELECT c.name, c.email, count(*) AS order_lines
FROM customers c JOIN orders o ON o.customer_id = c.id JOIN order_items oi ON oi.order_id = o.id
GROUP BY c.id ORDER BY c.id;
DELETE FROM order_items WHERE order_id = 2;     -- Bob's order lines go, Bob and products stay
SELECT (SELECT count(*) FROM customers) AS customers, (SELECT count(*) FROM products) AS products;

name  | email             | order_lines
------+-------------------+------------
Alice | alice@new.example | 3          
Bob   | bob@example.com   | 1          
(2 rows)

customers | products
----------+---------
2         | 3       
(1 row)



## 10. Keys: natural vs surrogate
- **Natural key**: a real-world value that identifies the row (email, ISBN, country code).
- **Surrogate key**: a meaningless generated id (`INTEGER PRIMARY KEY`).
- Natural keys can change (emails change, ISBNs get reissued) and then every foreign key must change too.
- Surrogate keys are small, stable, and fast to join on. Cost: they have no meaning, so ALSO put a `UNIQUE` on the natural key.
- Short stable codes (country `'IN'`, currency `'INR'`) are fine as natural primary keys.
- In SQLite, `INTEGER PRIMARY KEY` is an alias of the rowid: very fast. See 05-DDL-And-Integrity/01-Constraints-And-Keys.

In [16]:
%%sql
CREATE TABLE users (
    id    INTEGER PRIMARY KEY,           -- surrogate
    email TEXT NOT NULL UNIQUE,          -- natural, still protected
    name  TEXT NOT NULL
);
CREATE TABLE posts (id INTEGER PRIMARY KEY, user_id INTEGER NOT NULL REFERENCES users(id), title TEXT NOT NULL);
INSERT INTO users VALUES (1, 'asha@old.example', 'Asha');
INSERT INTO posts VALUES (1, 1, 'Hello'), (2, 1, 'Again');
UPDATE users SET email = 'asha@new.example' WHERE id = 1;       -- email changed, posts untouched
SELECT u.email, count(*) AS posts FROM users u JOIN posts p ON p.user_id = u.id GROUP BY u.id;

email            | posts
-----------------+------
asha@new.example | 2    
(1 row)



## 11. One-to-many
- One parent, many children: a customer has many orders, a department has many employees.
- Put the foreign key on the MANY side (`orders.customer_id`). Never store a list of order ids on the customer.
- Make the foreign key `NOT NULL` when a child must have a parent.
- Choose the delete rule on purpose: `ON DELETE CASCADE` (children go too), `RESTRICT` (block deletion), `SET NULL`.
- Add an index on the foreign key column: joins and cascades use it. SQLite does NOT create it automatically.

In [17]:
%%sql
CREATE INDEX idx_orders_customer ON orders(customer_id);
SELECT c.name, count(o.id) AS orders FROM customers c LEFT JOIN orders o ON o.customer_id = c.id GROUP BY c.id ORDER BY c.id;

name  | orders
------+-------
Alice | 2     
Bob   | 1     
(2 rows)



## 12. One-to-one
- Each row of A has at most one row of B, and vice versa: a user and their profile, an employee and a parking permit.
- Model it as a foreign key that is ALSO `UNIQUE` (or make it the primary key of the second table).
- Why split at all? Rarely-used or large columns, optional data, different permissions. If every row always has both, one table is simpler.

In [18]:
%%sql
CREATE TABLE user_profile (
    user_id INTEGER PRIMARY KEY REFERENCES users(id) ON DELETE CASCADE,   -- PK + FK = at most one profile per user
    bio     TEXT
);
INSERT INTO user_profile VALUES (1, 'Loves SQL');
SELECT u.name, p.bio FROM users u LEFT JOIN user_profile p ON p.user_id = u.id ORDER BY u.id;

name | bio      
-----+----------
Asha | Loves SQL
(1 row)



In [19]:
%%sql expect_error
-- A second profile for the same user is rejected
INSERT INTO user_profile VALUES (1, 'Second profile');

ERROR: UNIQUE constraint failed: user_profile.user_id



## 13. Many-to-many and bridge tables
- Students take many courses; a course has many students. You cannot put a foreign key on either side.
- Use a bridge (junction) table with two foreign keys; its primary key is the PAIR so the same link cannot be stored twice.
- The bridge can carry facts about the relationship itself (grade, enrolled_on, role). If it does, give it a good name (`enrolment`, not `student_course`).
- Many-to-many with no extra data is still a table, never a comma list.

In [20]:
%%sql
CREATE TABLE student (id INTEGER PRIMARY KEY, name TEXT NOT NULL);
CREATE TABLE course  (id INTEGER PRIMARY KEY, title TEXT NOT NULL UNIQUE);
CREATE TABLE enrolment (
    student_id  INTEGER NOT NULL REFERENCES student(id) ON DELETE CASCADE,
    course_id   INTEGER NOT NULL REFERENCES course(id)  ON DELETE CASCADE,
    enrolled_on TEXT NOT NULL,
    grade       TEXT,
    PRIMARY KEY (student_id, course_id)
);
INSERT INTO student VALUES (1, 'Asha'), (2, 'Ben');
INSERT INTO course VALUES (1, 'Maths'), (2, 'Art');
INSERT INTO enrolment VALUES (1, 1, '2025-01-10', 'A'), (1, 2, '2025-01-11', NULL), (2, 1, '2025-01-12', 'B');
SELECT s.name, c.title, e.grade FROM enrolment e JOIN student s ON s.id = e.student_id JOIN course c ON c.id = e.course_id ORDER BY s.name, c.title;

name | title | grade
-----+-------+------
Asha | Art   | NULL 
Asha | Maths | A    
Ben  | Maths | B    
(3 rows)



In [21]:
%%sql
-- How many courses does each student take? (LEFT JOIN + GROUP BY, see 02-Querying/02-Aggregation-And-Grouping)
SELECT s.name, count(e.course_id) AS courses FROM student s LEFT JOIN enrolment e ON e.student_id = s.id GROUP BY s.id ORDER BY s.name;

name | courses
-----+--------
Asha | 2      
Ben  | 1      
(2 rows)



## 14. Naming conventions
- Pick a style and keep it everywhere. Common, readable choice: `snake_case`, lowercase.
- Tables: singular (`customer`) or plural (`customers`), but be consistent. Bridge tables: name the relationship (`enrolment`).
- Primary key: `id`. Foreign key: `<table>_id` (`customer_id`). Booleans: `is_active`. Dates: `created_at`, `placed_on`.
- Avoid reserved words (`order`, `group`, `user` in some databases), spaces, and abbreviations nobody knows.
- Put units in names when it helps: `price_cents`, `weight_kg`, `timeout_ms`.
- Constraint and index names help error messages: `idx_<table>_<cols>`, `uq_<table>_<cols>`.

## 15. Storing money: integer cents
- Never `REAL`/`FLOAT` for money. Binary floating point cannot store 0.1 exactly, and the errors add up.
- Store the smallest unit as an INTEGER (cents, paise). Show it as `cents / 100.0` only when displaying.
- Add a currency column if you handle more than one currency. Never add amounts in different currencies.
- Percentages: multiply first, divide last (`amount * 18 / 100`) and decide the rounding rule on purpose.

In [22]:
%%sql
SELECT 0.1 + 0.2 AS float_sum, 0.1 + 0.2 = 0.3 AS float_equal, 10 + 20 AS cents_sum, 10 + 20 = 30 AS cents_equal;

float_sum           | float_equal | cents_sum | cents_equal
--------------------+-------------+-----------+------------
0.30000000000000004 | 0           | 30        | 1          
(1 row)



In [23]:
%%sql
CREATE TABLE invoice (id INTEGER PRIMARY KEY, amount_cents INTEGER NOT NULL CHECK (amount_cents >= 0), currency TEXT NOT NULL DEFAULT 'INR');
INSERT INTO invoice (amount_cents) VALUES (19999), (5001);
SELECT sum(amount_cents) AS total_cents, printf('%.2f', sum(amount_cents) / 100.0) AS total_display FROM invoice;

total_cents | total_display
------------+--------------
25000       | 250.00       
(1 row)



## 16. Storing dates and times: ISO-8601 text
- SQLite has no date type. Store text like `'2025-03-01'` or `'2025-03-01 14:30:00'` (or `...T14:30:00Z` for UTC).
- ISO-8601 text sorts correctly as plain text and works with `date()`, `datetime()`, `strftime()`.
- Store times in UTC; convert to local time only when displaying.
- Alternatives: Unix integers (compact, not readable) or Julian day numbers. Pick ONE per column and add a CHECK.
- Bad: `'1/3/25'` (is that 1 March or 3 January?) sorts wrongly and cannot be parsed reliably.

In [24]:
%%sql
CREATE TABLE event (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    starts_at TEXT NOT NULL CHECK (datetime(starts_at) IS NOT NULL)   -- rejects text SQLite cannot read as a date
);
INSERT INTO event (name, starts_at) VALUES ('Launch', '2025-03-01 09:00:00'), ('Review', '2025-02-15 16:30:00');
SELECT name, starts_at, strftime('%Y-%m', starts_at) AS month FROM event ORDER BY starts_at;

name   | starts_at           | month  
-------+---------------------+--------
Review | 2025-02-15 16:30:00 | 2025-02
Launch | 2025-03-01 09:00:00 | 2025-03
(2 rows)



In [25]:
%%sql expect_error
INSERT INTO event (name, starts_at) VALUES ('Bad', 'next tuesday');

ERROR: CHECK constraint failed: datetime(starts_at) IS NOT NULL



## 17. Enums: lookup table vs CHECK
- A column with a small fixed set of values (`status`: pending, paid, shipped).
- **CHECK** (`CHECK (status IN ('pending','paid','shipped'))`): simplest, no extra table. Changing the list needs a table rebuild in SQLite (see 01-Data-Modification section 18).
- **Lookup table** + foreign key: values live as data. Add a new status with an INSERT. You can attach extra columns (label, sort order, is_final). Best when the list changes or has meaning of its own.
- Rule of thumb: stable and tiny: CHECK. Business-managed or growing: lookup table.

In [26]:
%%sql
CREATE TABLE order_status (code TEXT PRIMARY KEY, label TEXT NOT NULL, is_final INTEGER NOT NULL DEFAULT 0);
INSERT INTO order_status VALUES ('pending', 'Waiting for payment', 0), ('paid', 'Paid', 0), ('shipped', 'On its way', 0), ('cancelled', 'Cancelled', 1);
CREATE TABLE purchase (id INTEGER PRIMARY KEY, status TEXT NOT NULL DEFAULT 'pending' REFERENCES order_status(code));
INSERT INTO purchase (status) VALUES ('paid'), ('pending');
SELECT p.id, s.label, s.is_final FROM purchase p JOIN order_status s ON s.code = p.status ORDER BY p.id;

id | label               | is_final
---+---------------------+---------
1  | Paid                | 0       
2  | Waiting for payment | 0       
(2 rows)



In [27]:
%%sql expect_error
INSERT INTO purchase (status) VALUES ('lost in space');

ERROR: FOREIGN KEY constraint failed



## 18. JSON columns: when and when not
- SQLite can store JSON as text and query it with `json_extract`, `->`, `->>` (3.38+), `json_each`.
- Good for: truly free-form data you rarely filter on (settings, API payloads, event details), or attributes that differ wildly per row.
- Bad for: anything you filter, join, sum, or constrain regularly. No foreign keys into JSON, no type checking, harder to index (you can index an expression).
- Rule: if you write `WHERE json_extract(...)` in many queries, it wants to be a real column or table.
- Add `CHECK (json_valid(col))` so garbage cannot be stored.

In [28]:
%%sql
CREATE TABLE device (id INTEGER PRIMARY KEY, name TEXT NOT NULL, settings TEXT CHECK (json_valid(settings)));
INSERT INTO device (name, settings) VALUES
    ('Lamp', '{"brightness": 80, "colour": "warm"}'),
    ('Fan',  '{"speed": 3}');
SELECT name, settings ->> '$.brightness' AS brightness, settings ->> '$.speed' AS speed FROM device ORDER BY id;

name | brightness | speed
-----+------------+------
Lamp | 80         | NULL 
Fan  | NULL       | 3    
(2 rows)



In [29]:
%%sql expect_error
INSERT INTO device (name, settings) VALUES ('Broken', '{not json');

ERROR: CHECK constraint failed: json_valid(settings)



## 19. Soft delete and audit columns
- Common housekeeping columns: `created_at`, `updated_at`, optionally `created_by`, `deleted_at`.
- Soft delete (`deleted_at` NULL = alive) keeps history and keeps foreign keys valid; see 01-Data-Modification section 17.
- Gotcha: UNIQUE constraints still count soft-deleted rows. Fix with a partial unique index: `CREATE UNIQUE INDEX ... WHERE deleted_at IS NULL`.
- `updated_at` is easiest kept correct by a trigger; set `created_at` with a DEFAULT. (Defaults like `CURRENT_TIMESTAMP` are fine in a table definition, we just avoid printing "now" in this notebook.)
- For full history, write changes to a separate audit table via triggers.

In [30]:
%%sql
CREATE TABLE site_member (
    id INTEGER PRIMARY KEY,
    email TEXT NOT NULL,
    created_at TEXT NOT NULL DEFAULT '2025-01-01',     -- fixed default only to keep this notebook repeatable
    deleted_at TEXT
);
CREATE UNIQUE INDEX uq_site_member_email_alive ON site_member(email) WHERE deleted_at IS NULL;
INSERT INTO site_member (email) VALUES ('a@example.com');
UPDATE site_member SET deleted_at = '2025-03-01' WHERE email = 'a@example.com';
INSERT INTO site_member (email) VALUES ('a@example.com');       -- same email can sign up again, the old row is soft-deleted
SELECT id, email, deleted_at FROM site_member ORDER BY id;

id | email         | deleted_at
---+---------------+-----------
1  | a@example.com | 2025-03-01
2  | a@example.com | NULL      
(2 rows)



In [31]:
%%sql expect_error
INSERT INTO site_member (email) VALUES ('a@example.com');       -- but two ALIVE rows with that email are refused

ERROR: UNIQUE constraint failed: site_member.email



## 20. When to denormalize on purpose
- Denormalize = store a fact in more than one place, knowingly, for speed or simplicity.
- Reasons: reporting tables read far more than they are written; expensive joins or aggregates; history snapshots (price at sale).
- Always keep ONE source of truth and a way to rebuild the copy. Update copies in the same transaction (see 02-Transactions) or with a trigger.
- Alternatives first: add an index, use a view, cache the result. Denormalize after measuring (see 05-DDL-And-Integrity/02-Views-And-Triggers).
- Example: store `order_total_cents` on the order so list pages do not sum all lines every time.

In [32]:
%%sql
ALTER TABLE orders ADD COLUMN total_cents INTEGER;   -- deliberate copy: derived from order_items
UPDATE orders SET total_cents = (SELECT COALESCE(SUM(quantity * price_at_sale), 0) FROM order_items WHERE order_id = orders.id);
SELECT id, total_cents FROM orders ORDER BY id;

id | total_cents
---+------------
1  | 3997       
2  | 0          
3  | 5997       
(3 rows)



In [33]:
%%sql
-- A cheap consistency check you can run any time: copy vs source of truth (should return no rows)
SELECT o.id FROM orders o
WHERE o.total_cents <> (SELECT COALESCE(SUM(quantity * price_at_sale), 0) FROM order_items WHERE order_id = o.id)
ORDER BY o.id;

id
--
(0 rows)



## 21. ER sketch as a text diagram
- An entity-relationship (ER) diagram shows tables and how they connect. `||` = exactly one, `o{` = zero or many, `|{` = one or many.
- Drawing it first, even in text, forces you to decide the relationships before writing SQL.
- Our small library design (next sections):
```text
MEMBER ||--o{ LOAN : borrows
BOOK   ||--|{ COPY : "has physical"
COPY   ||--o{ LOAN : "is lent in"
BOOK   }o--o{ AUTHOR : "written by (book_author)"
BOOK   }o--|| PUBLISHER : "published by"

 +-----------+       +-------------+       +----------+
 | publisher |1-----*|    book     |*-----*|  author  |
 +-----------+       +-------------+  via  +----------+
                           |1          book_author
                           |
                           *
                      +---------+ 1     * +----------+ *     1 +--------+
                      |  copy   |--------|   loan   |---------| member |
                      +---------+        +----------+         +--------+
```

## 22. Worked design: a small library, from requirements
- Requirements: (1) a book has a title, ISBN, one publisher, and one or more authors; (2) the library owns several physical copies of a book; (3) members borrow copies; each loan has a borrow date, a due date and (when returned) a return date; (4) a copy cannot be lent to two people at once; (5) members have a unique email.
- Step 1, nouns become tables: publisher, author, book, copy, member, loan.
- Step 2, relationships: book-publisher one-to-many; book-author many-to-many (bridge `book_author`); book-copy one-to-many; copy-loan one-to-many; member-loan one-to-many.
- Step 3, rules become constraints: ISBN unique, dates as ISO text, due date not before borrow date, one open loan per copy (partial unique index on `copy_id WHERE returned_on IS NULL`).

In [34]:
%%sql
CREATE TABLE publisher (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE);
CREATE TABLE author    (id INTEGER PRIMARY KEY, name TEXT NOT NULL);
CREATE TABLE book (
    id           INTEGER PRIMARY KEY,
    isbn         TEXT NOT NULL UNIQUE,
    title        TEXT NOT NULL,
    publisher_id INTEGER NOT NULL REFERENCES publisher(id)
);
CREATE TABLE book_author (
    book_id   INTEGER NOT NULL REFERENCES book(id) ON DELETE CASCADE,
    author_id INTEGER NOT NULL REFERENCES author(id),
    position  INTEGER NOT NULL DEFAULT 1,           -- first author, second author...
    PRIMARY KEY (book_id, author_id)
);
CREATE TABLE copy (
    id      INTEGER PRIMARY KEY,
    book_id INTEGER NOT NULL REFERENCES book(id),
    barcode TEXT NOT NULL UNIQUE
);
CREATE TABLE member (id INTEGER PRIMARY KEY, name TEXT NOT NULL, email TEXT NOT NULL UNIQUE);
CREATE TABLE loan (
    id          INTEGER PRIMARY KEY,
    copy_id     INTEGER NOT NULL REFERENCES copy(id),
    member_id   INTEGER NOT NULL REFERENCES member(id),
    borrowed_on TEXT NOT NULL,
    due_on      TEXT NOT NULL,
    returned_on TEXT,
    CHECK (due_on >= borrowed_on),
    CHECK (returned_on IS NULL OR returned_on >= borrowed_on)
);
CREATE UNIQUE INDEX uq_loan_open_copy ON loan(copy_id) WHERE returned_on IS NULL;   -- one open loan per copy
CREATE INDEX idx_loan_member ON loan(member_id);
SELECT name FROM sqlite_master WHERE type = 'table' AND name IN ('publisher','author','book','book_author','copy','member','loan') ORDER BY name;

name       
-----------
author     
book       
book_author
copy       
loan       
member     
publisher  
(7 rows)



## 23. Worked design: load data and ask the business questions
- Seed a few rows, then answer real questions with joins. A good schema makes each question a short query.
- See 02-Querying/01-Joins and 02-Querying/02-Aggregation-And-Grouping for the query techniques.

In [35]:
%%sql
INSERT INTO publisher VALUES (1, 'Orbit Press'), (2, 'Lakeside');
INSERT INTO author VALUES (1, 'N. Rao'), (2, 'L. Chen'), (3, 'P. Das');
INSERT INTO book VALUES (1, '978-0-00-000001-1', 'Learning SQL', 1), (2, 'ISBN-0002', 'Data Design', 2), (3, 'ISBN-0003', 'Shared Work', 1);
INSERT INTO book_author VALUES (1, 1, 1), (2, 2, 1), (3, 1, 1), (3, 3, 2);
INSERT INTO copy VALUES (1, 1, 'BC-001'), (2, 1, 'BC-002'), (3, 2, 'BC-003'), (4, 3, 'BC-004');
INSERT INTO member VALUES (1, 'Asha', 'asha@example.com'), (2, 'Ben', 'ben@example.com');
INSERT INTO loan (copy_id, member_id, borrowed_on, due_on, returned_on) VALUES
    (1, 1, '2025-01-05', '2025-01-19', '2025-01-15'),
    (1, 2, '2025-02-01', '2025-02-15', NULL),
    (3, 1, '2025-02-03', '2025-02-17', NULL);
-- Books with all their authors, in author order
SELECT b.title, group_concat(a.name, ', ' ORDER BY ba.position) AS authors   -- ORDER BY inside group_concat needs SQLite 3.44+
FROM book b
JOIN book_author ba ON ba.book_id = b.id
JOIN author a ON a.id = ba.author_id
GROUP BY b.id ORDER BY b.title;

title        | authors       
-------------+---------------
Data Design  | L. Chen       
Learning SQL | N. Rao        
Shared Work  | N. Rao, P. Das
(3 rows)



In [36]:
%%sql
-- Which copies are on loan right now, and who has them?
SELECT b.title, c.barcode, m.name AS borrower, l.due_on
FROM loan l JOIN copy c ON c.id = l.copy_id JOIN book b ON b.id = c.book_id JOIN member m ON m.id = l.member_id
WHERE l.returned_on IS NULL ORDER BY l.due_on, c.barcode;

title        | barcode | borrower | due_on    
-------------+---------+----------+-----------
Learning SQL | BC-001  | Ben      | 2025-02-15
Data Design  | BC-003  | Asha     | 2025-02-17
(2 rows)



In [37]:
%%sql
-- Copies available per book (total copies minus open loans)
SELECT b.title, count(c.id) AS total_copies,
       count(c.id) - count(l.id) AS available
FROM book b
JOIN copy c ON c.book_id = b.id
LEFT JOIN loan l ON l.copy_id = c.id AND l.returned_on IS NULL
GROUP BY b.id ORDER BY b.title;

title        | total_copies | available
-------------+--------------+----------
Data Design  | 1            | 0        
Learning SQL | 2            | 1        
Shared Work  | 1            | 1        
(3 rows)



In [38]:
%%sql expect_error
-- The design enforces the rule "one open loan per copy": lending copy 1 again while Ben has it is refused
INSERT INTO loan (copy_id, member_id, borrowed_on, due_on) VALUES (1, 1, '2025-02-05', '2025-02-19');
-- A due date before the borrow date is refused too
INSERT INTO loan (copy_id, member_id, borrowed_on, due_on) VALUES (2, 1, '2025-02-05', '2025-01-01');

ERROR: UNIQUE constraint failed: loan.copy_id

ERROR: CHECK constraint failed: due_on >= borrowed_on



## 24. Design checklist
- Can I name the key of every table? Does every other column depend on the whole key and nothing but the key?
- Is every repeated fact stored once (or copied deliberately, with a note and a check query)?
- Are all relationships foreign keys, with a chosen delete rule and an index on the FK column?
- Are money, dates and enums stored with the patterns above, and guarded by CHECKs?
- Are the business rules ("one open loan per copy") in the database as constraints, not only in app code?
- Is there a plan for history (soft delete, audit table) and for changing the schema later (migrations in a transaction, see 06-Data-Modification/02-Transactions)?